In [ ]:
import fiftyone.zoo as foz

dataset = foz.load_zoo_dataset(
    "coco-2017",
    #splits=["train", "validation", "test"],
    splits=["train"],
    label_types=["detections"],
    classes=["chair"],
    max_samples=10000,
)


In [ ]:
print(dataset)

In [ ]:
dataset.compute_metadata()

In [ ]:
import torch
from ultralytics import YOLO
from PIL import Image
import numpy as np
import fiftyone as fo

# Get class list
classes = dataset.default_classes

# Run the model on GPU if it is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model (not just the path)
model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/doors/best.pt")
model.to(device)

print("Model ready")

# Add predictions to samples
with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        # Load image as PIL Image (don't convert to tensor)
        image = Image.open(sample.filepath)
        
        # Perform inference directly on PIL image
        results = model(image)
        
        # Extract predictions from results
        result = results[0]
        boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy().astype(int)
        
        # Get image dimensions
        w, h = image.size
        
        # Convert detections to FiftyOne format
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            # Convert to [top-left-x, top-left-y, width, height]
            # in relative coordinates in [0, 1] x [0, 1]
            x1, y1, x2, y2 = box
            rel_box = [x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h]

            detections.append(
                fo.Detection(
                    label=model.names[label],  # Use model's class names
                    bounding_box=rel_box,
                    confidence=score
                )
            )

        # Save predictions to dataset
        sample["predictions"] = fo.Detections(detections=detections)
        sample.save()

# Filter samples that have "Door" predictions
door_predictions_view = dataset.filter_labels("predictions", fo.ViewField("label") == "Door")

# Get the sample IDs that contain Door predictions
door_sample_ids = list(door_predictions_view.values("id"))

# Delete these samples from the dataset
dataset.delete_samples(door_sample_ids)

print(f"Deleted {len(door_sample_ids)} samples with Door predictions")

# Save the changes
dataset.save()

In [ ]:
# More efficient approach using bulk updates
import numpy as np

# Create split labels
total = len(dataset)
splits = (["train"] * int(0.7 * total) + 
          ["val"] * int(0.2 * total) + 
          ["test"] * (total - int(0.7 * total) - int(0.2 * total)))

# Shuffle splits
np.random.shuffle(splits)

# Assign splits to samples
dataset.set_values("split", splits)

print(f"Split created: {splits.count('train')} train, {splits.count('val')} val, {splits.count('test')} test")

In [ ]:

import fiftyone.utils.random as four

four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

In [ ]:
# See unique values in the split field
print("Split values:", dataset.distinct("split"))

# Count samples per split
split_counts = dataset.count_values("split")
print("Split counts:", split_counts)

In [ ]:
# Import YOLO dataset from YAML file
import fiftyone as fo

name="ebi-rundgang-z1fgl-1-dataset"
# Option 1: Delete existing dataset first
try:
    fo.delete_dataset(name)
except:
    pass

# Load the YOLO dataset
dataset2 = fo.Dataset.from_dir(
    dataset_dir="/mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-rundgang-z1fgl-1-yolov5",
    dataset_type=fo.types.YOLOv5Dataset,
    name=name
)

print(f"Loaded dataset2: {dataset2}")
print(f"Number of samples: {len(dataset2)}")
print(f"Classes: {dataset2.default_classes}")

In [ ]:
# Debug the dataset directory structure
import os

dataset_dir = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-rundgang-z1fgl-1-yolov5"

print("Checking directory structure:")
print(f"Directory exists: {os.path.exists(dataset_dir)}")

if os.path.exists(dataset_dir):
    print("\nContents:")
    for item in os.listdir(dataset_dir):
        item_path = os.path.join(dataset_dir, item)
        if os.path.isdir(item_path):
            print(f"📁 {item}/")
            # Show first few items in subdirectories
            sub_items = os.listdir(item_path)[:5]
            for sub_item in sub_items:
                print(f"   {sub_item}")
            if len(os.listdir(item_path)) > 5:
                print(f"   ... and {len(os.listdir(item_path)) - 5} more")
        else:
            print(f"📄 {item}")

# Check for YAML files
yaml_files = [f for f in os.listdir(dataset_dir) if f.endswith('.yaml')]
print(f"\nYAML files found: {yaml_files}")

if yaml_files:
    import yaml
    yaml_path = os.path.join(dataset_dir, yaml_files[0])
    with open(yaml_path, 'r') as f:
        yaml_content = yaml.safe_load(f)
    print(f"\nYAML content:")
    print(yaml_content)

In [ ]:
# Visualize the dataset in the FiftyOne App
import fiftyone as fo
session = fo.launch_app(dataset2)

In [ ]:


export_dir = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test"
#label_field = "ground_truth"  # for example

# The splits to export
splits = ["train", "test","val"]

# All splits must use the same classes list
classes = ["chair"]

# Export the splits
for split in splits:
    split_view = dataset.match_tags(split)
    split_view.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=label_field,
        split=split,
        classes=classes,
    )